Input Boss Name

Get boss's data from db

seperate data into decks with 6+ wins, and decks with less

Create a dictionary for each set of performers

sum up all the cards from each deck into their respective dicts

Take the average of everything

In [1]:

from collections import defaultdict

import src.card_data_collection_pipeline as card_data_collection_pipeline
import src.db_operations as db_operations
import src.helpers as helpers

import pandas as pd


In [2]:

def db_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return db_operations.find_first_in_table('main_table', 'card_data', {'id':id})


def url_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return card_data_collection_pipeline.add_card_info_to_db(id)
    print(id)
    raise NotImplementedError 


In [3]:
# BOSS_NAME = "Deeplands, Reguregnus"
BOSS_NAME = "Sugary and Scary Land, Heartluru"

In [4]:
full_df = db_operations.get_answer_from_table(database='main_table',
                                              table='all_events',
                                              query={'boss':
                                                     {'$regex':BOSS_NAME}
                                                     }
                                             )

In [5]:
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,regalis_piece,crit_heal_count,draw_count,stand_heal_count
0,69b018213efa47b42b6c188f,6,BigDaddyBrie,"Sugary and Scary Land, Heartluru",8,Dark States,741C4,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Union the Sky,0.0,1.0,1.0
1,69b018213efa47b42b6c18b2,41,Beagle-Man,"Sugary and Scary Land, Heartluru",5,Dark States,3N440,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",None,0.0,0.0,2.0
2,69b018213efa47b42b6c18ec,99,Padding Numbers,"Sugary and Scary Land, Heartluru",4,Dark States,6TCSY,{'RideDeck': {'DZ-BT09/004EN : Sugary and Scar...,Rosemont,NA,"October 4, 2025",Sound of Wave,1.0,1.0,0.0
3,69b018213efa47b42b6c18f0,103,DSKLucius,"Sugary and Scary Land, Heartluru",4,Dark States,25Q1A,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Protection: Twincast,0.0,1.0,2.0
4,69b018213efa47b42b6c18f6,109,CSRaccoon,"Sugary and Scary Land, Heartluru",4,Dark States,20P0U,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Radiance Caliburn,0.0,1.0,0.0


In [6]:
# #For Alicia.
# # Count amt of cat in deck
# # cat_df = full_df
# # cat_df.loc[:, "cat_count"] = cat_df.deck.apply(
#     # lambda deck: helpers.card_type_in_deck(
#     # deck, ["Purple Jeweled Beast, Almethys"]
# # ))
# # cat_df.groupby('cat_count').describe()

#todo why broken???
# def card_amt_for_boss(df=None, bossname=None, cardname=None):
#     if (type(df) == type(None)) & (bossname != None):
#         foo = db_operations.get_answer_from_table(query=
#                                                  {'boss':
#                                                     {'$regex':BOSS_NAME}
#                                                  })
#     foo=pd.DataFrame()
#     foo = df.apply(
#         lambda deck: helpers.card_type_in_deck(
#             deck, [cardname]
#         ))
#     return foo
#     # return foo.groupby(TODO).describe()

# cat_df =card_amt_for_boss(full_df, 
#                           "Sugary and Scary Land, Heartluru",
#                           "Purple Jeweled Beast, Almethys")
# cat_df

Currently, I'm noticing two bugs with the system.
One: Alphnights keeps getting added to the db again and again, so it's failing to read??
Also noticed maybe the same thing with hakasui and remred?

Two: Most cards with no flavor text have an issue with the format~artist 'line'

In [7]:
# lets try just saving and re loading one copy of alphknights. If we can do one, we can do more :)
# memoizer = memoizer or None
# memoizer

In [8]:
memoizer = dict()

In [9]:
# memoizer = memoizer or dict()
copy = full_df
for i, row in full_df.iterrows():
    new_deck = defaultdict(int)
    for card_id_and_name, amount in row.deck['MainDeck'].items():
        # print(card_id_and_name,'\n', amount,'\n~~~~~~~~~~~~~~~~~~~~~~')
        # if type(amount) != type(67):
        #     print(type(amount))
        id = helpers.extract_card_id(card_id_and_name)
        try:
            if memoizer[id]:
                lookup = memoizer[id]
        except KeyError:
            lookup = None

        if not lookup:
            lookup = db_lookup(card_id_and_name)
            memoizer[id] = lookup
        if not lookup:
            lookup = url_lookup( card_id_and_name)
            memoizer[id] = lookup

        name = lookup['name']

        if 'Trigger' in lookup['type']:
            if 5000 == lookup['power']:
                name = 'Vanilla Trigger'

        if '[CONT]:Sentinel' in lookup['effect']:
            if 'Unit' in lookup['type']:
                name = 'Sentinel'
            else:
                name = 'Elementaria'

        if '[CONT]:This card is' in lookup['effect']:
            first = lookup['effect'].find('"')+1
            second = lookup['effect'].find('"', first)
            name = lookup['effect'][first:second]

        new_deck[name] += amount
    
    copy.at[i, 'deck']['MainDeck'] = new_deck

In [10]:
# foo = card_data_collection_pipeline.get_card_info_from_bushi_site('DZ-SS08/007EN')
# foo

In [11]:
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,regalis_piece,crit_heal_count,draw_count,stand_heal_count
0,69b018213efa47b42b6c188f,6,BigDaddyBrie,"Sugary and Scary Land, Heartluru",8,Dark States,741C4,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Union the Sky,0.0,1.0,1.0
1,69b018213efa47b42b6c18b2,41,Beagle-Man,"Sugary and Scary Land, Heartluru",5,Dark States,3N440,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",None,0.0,0.0,2.0
2,69b018213efa47b42b6c18ec,99,Padding Numbers,"Sugary and Scary Land, Heartluru",4,Dark States,6TCSY,{'RideDeck': {'DZ-BT09/004EN : Sugary and Scar...,Rosemont,NA,"October 4, 2025",Sound of Wave,1.0,1.0,0.0
3,69b018213efa47b42b6c18f0,103,DSKLucius,"Sugary and Scary Land, Heartluru",4,Dark States,25Q1A,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Protection: Twincast,0.0,1.0,2.0
4,69b018213efa47b42b6c18f6,109,CSRaccoon,"Sugary and Scary Land, Heartluru",4,Dark States,20P0U,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Radiance Caliburn,0.0,1.0,0.0


In [12]:
WINS = 5
top_df = full_df[full_df['wins'] > WINS]
bot_df = full_df[full_df['wins'] <= WINS]

In [13]:

top_dict = defaultdict(int)
bot_dict = defaultdict(int)
all_dict = defaultdict(int)

for dict in top_df.deck:
    for k, v in dict['MainDeck'].items():
        top_dict[k] += v
        all_dict[k] += v

for dict in bot_df.deck:
    for k, v in dict['MainDeck'].items():
        bot_dict[k] += v
        all_dict[k] += v
        

In [14]:
num_tops = len(top_df)
num_bots = len(bot_df)
num_total = num_tops + num_bots

print("Number of Tops: ", num_tops)
print("Number of Bots: ", num_bots)
print("Total Number: ", num_total)

Number of Tops:  105
Number of Bots:  628
Total Number:  733


In [15]:
top_dict = {k: v / num_tops for k,v in top_dict.items()}
bot_dict = {k: v / num_bots for k,v in bot_dict.items()}
all_dict = {k: v / num_total for k,v in all_dict.items()}

top_dict = {k:v for k,v in sorted(top_dict.items(), key=lambda k: k[1], reverse=True)}
bot_dict = {k:v for k,v in sorted(bot_dict.items(), key=lambda k: k[1], reverse=True)}
all_dict = {k:v for k,v in sorted(all_dict.items(), key=lambda k: k[1], reverse=True)}

In [16]:
top_dict

{'Vanilla Trigger': 9.523809523809524,
 'Toys Monster, Blavyloss': 4.0,
 'Sparkly Bunny': 4.0,
 'Toys Are Made for Children': 4.0,
 'Giant Fluffy': 3.9238095238095236,
 'Block Horse': 3.6095238095238096,
 'Sentinel': 3.0476190476190474,
 'Diabolos Girls, Natalia': 3.0095238095238095,
 'Sugary and Scary Land, Heartluru': 3.0,
 'Avarice Wrester': 1.9333333333333333,
 'Incorruptible Holy Light, Eufha': 1.9238095238095239,
 'Opener of Heart, Philya': 1.3904761904761904,
 'Stem Deviate Dragon': 1.2857142857142858,
 'Purple Jeweled Beast, Almethys': 1.1047619047619048,
 'Elementaria': 0.9238095238095239,
 'Steam Gunner, Tizkar': 0.8285714285714286,
 'Protection: Twincast': 0.8285714285714286,
 'Fashion Doll, Ruby Rouge': 0.6857142857142857,
 'Illusory Lantern Demonic Lady, Forcalor': 0.2857142857142857,
 'Depravity Funeral Judgment, Valack': 0.24761904761904763,
 'Spiritual King of Ignition, Valnout': 0.1619047619047619,
 'Fire Regalis': 0.12380952380952381,
 'Brilliant Floral, Uania': 0.085

In [17]:
bot_dict

{'Vanilla Trigger': 10.0828025477707,
 'Toys Are Made for Children': 4.0,
 'Toys Monster, Blavyloss': 3.9936305732484074,
 'Sparkly Bunny': 3.9856687898089174,
 'Giant Fluffy': 3.9187898089171975,
 'Block Horse': 3.7261146496815285,
 'Sentinel': 3.0605095541401273,
 'Sugary and Scary Land, Heartluru': 2.9968152866242037,
 'Diabolos Girls, Natalia': 2.9156050955414012,
 'Avarice Wrester': 2.0429936305732483,
 'Incorruptible Holy Light, Eufha': 1.4458598726114649,
 'Opener of Heart, Philya': 1.4028662420382165,
 'Stem Deviate Dragon': 1.2914012738853504,
 'Purple Jeweled Beast, Almethys': 1.0302547770700636,
 'Elementaria': 0.9203821656050956,
 'Protection: Twincast': 0.8821656050955414,
 'Fashion Doll, Ruby Rouge': 0.6353503184713376,
 'Steam Gunner, Tizkar': 0.6226114649681529,
 'Depravity Funeral Judgment, Valack': 0.3200636942675159,
 'Illusory Lantern Demonic Lady, Forcalor': 0.22611464968152867,
 'Brilliant Floral, Uania': 0.10031847133757962,
 'Spiritual King of Ignition, Valnout'

In [18]:
all_dict

{'Vanilla Trigger': 10.002728512960436,
 'Toys Are Made for Children': 4.0,
 'Toys Monster, Blavyloss': 3.9945429740791267,
 'Sparkly Bunny': 3.9877216916780354,
 'Giant Fluffy': 3.9195088676671213,
 'Block Horse': 3.709413369713506,
 'Sentinel': 3.058663028649386,
 'Sugary and Scary Land, Heartluru': 2.9972714870395634,
 'Diabolos Girls, Natalia': 2.9290586630286493,
 'Avarice Wrester': 2.0272851296043655,
 'Incorruptible Holy Light, Eufha': 1.514324693042292,
 'Opener of Heart, Philya': 1.4010914051841745,
 'Stem Deviate Dragon': 1.290586630286494,
 'Purple Jeweled Beast, Almethys': 1.0409276944065484,
 'Elementaria': 0.9208731241473397,
 'Protection: Twincast': 0.8744884038199181,
 'Steam Gunner, Tizkar': 0.6521145975443383,
 'Fashion Doll, Ruby Rouge': 0.6425648021828103,
 'Depravity Funeral Judgment, Valack': 0.3096862210095498,
 'Illusory Lantern Demonic Lady, Forcalor': 0.23465211459754434,
 'Spiritual King of Ignition, Valnout': 0.1009549795361528,
 'Brilliant Floral, Uania': 0

In [19]:
diff_dict = defaultdict(int)
for k, v in bot_dict.items():
    try:
        diff_dict[k] = top_dict[k] - v
    except:
        continue

diff_dict = {k:v for k,v in sorted(diff_dict.items(), key=lambda k: k[1], reverse=True)}

In [20]:
diff_dict

{'Incorruptible Holy Light, Eufha': 0.477949651198059,
 'Steam Gunner, Tizkar': 0.20595996360327573,
 'Diabolos Girls, Natalia': 0.09391871398240825,
 'Fire Regalis': 0.0824082499241735,
 'Purple Jeweled Beast, Almethys': 0.0745071276918412,
 'Spiritual King of Ignition, Valnout': 0.07114043069457084,
 'Illusory Lantern Demonic Lady, Forcalor': 0.05959963603275703,
 'Fashion Doll, Ruby Rouge': 0.050363967242948116,
 'Sparkly Bunny': 0.014331210191082633,
 'Elementaria Sanctitude': 0.009463148316651501,
 'Gratias Gradale': 0.007931452835911436,
 'Toys Monster, Blavyloss': 0.006369426751592577,
 'Giant Fluffy': 0.00501971489232611,
 'Recordinguom Parrot': 0.004746739460115257,
 'Elementaria': 0.003427358204428299,
 'Sugary and Scary Land, Heartluru': 0.0031847133757962887,
 'Evergreen Transphere': 0.0031543827722171677,
 'Toys Are Made for Children': 0.0,
 'Union the Sky': -3.0330603579009977e-05,
 'Stem Deviate Dragon': -0.005686988171064611,
 'Opener of Heart, Philya': -0.0123900515620

Ah! I noticed an issue where cards not being in both dicts causes an issue. This sorta ruins the tool, 